# Offline Learning on Different Replay Buffer Compositions

In [83]:
import torch
import numpy as np
from omegaconf import OmegaConf
from functools import partial
import gymnasium as gym
import matplotlib.pyplot as plt
import re
from pathlib import Path
from tqdm.notebook import tqdm

import time
import datetime
import os

import bbrl_utils
from bbrl_utils.notebook import setup_tensorboard
from bbrl.stats import WelchTTest
from bbrl.agents import Agent, Agents, TemporalAgent
from bbrl.agents.gymnasium import ParallelGymAgent, make_env
from bbrl.workspace import Workspace
from bbrl.utils.replay_buffer import ReplayBuffer

import bbrl_gymnasium

from pmind.algorithms import DQN, DDPG, TD3, OfflineTD3
from pmind.losses import dqn_compute_critic_loss, ddqn_compute_critic_loss
from pmind.training import (
    run_dqn,
    run_ddpg,
    run_td3,
)
from pmind.replay import (
    collect_policy_transitions,
    collect_uniform_transitions,
    mix_transitions,
    test_rb_uniform_proportions,
    test_rb_noise_levels
)

from pmind.visualization import plot_perf_vs_rb_composition_from_dict

from pmind.config.loader import load_config

bbrl_utils.setup()

OUTPUT_DIR = f"../results/test_rb_compositions-{datetime.datetime.now().strftime('%Y-%m-%d-%H%M%S')}"
os.mkdir(OUTPUT_DIR)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [84]:
ENV_NAMES = (
    # "CartPoleContinuous-v1",
    # "Pendulum-v1",
    "MountainCarContinuous-v0",
    # "LunarLanderContinuous-v3",
)
BUFFER_SIZE = 200_000
START_WITH_BEST = False
FIRST_ONLY = True
TEST_UNIFORM_PROPORTIONS = True
TEST_NOISE_LEVELS = False
PROPORTIONS = [0.5] #[0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
SEEDS = [101] #[0, 1, 2, 3, 4]
# NOTE: set small values for test:
MAX_STEPS = 10_000 #100_000
NB_EVAL_ENVS = 3 #10
EVAL_INTERVAL = 10 #100
SAVE_RB_POLICY_INTERVAL = 2_000
BATCH_SIZE = 10 #64
NN_ARCHITECTURE = [32,16] #[400,300]

In [85]:
def load_rb_files(directory):
    directory = Path(directory)
    pattern = re.compile(r"^rb-(-?\d+)\.pt$")
    
    result = {}

    for file in directory.iterdir():
        if file.is_file():
            match = pattern.match(file.name)
            if match:
                a_number = int(match.group(1))
                content = torch.load(file, weights_only=False)
                result[a_number] = content

    return result

for ENV_NAME in ENV_NAMES:
    rb_exploit = load_rb_files(f"../models/{ENV_NAME}/")
    print(rb_exploit)
    torch.save(rb_exploit, f"../models/{ENV_NAME}/rb-exploit.pt")

{35: <bbrl.utils.replay_buffer.ReplayBuffer object at 0x1315d6d70>, 86: <bbrl.utils.replay_buffer.ReplayBuffer object at 0x1315d4400>, 61: <bbrl.utils.replay_buffer.ReplayBuffer object at 0x1315d69b0>, 82: <bbrl.utils.replay_buffer.ReplayBuffer object at 0x1315d7310>}


In [89]:
REWARDS_TO_TAKE = [61]#[35, 61, 86]

for ENV_NAME in ENV_NAMES:
    print(f"Testing {ENV_NAME}:")
    cfg = load_config("td3")
    cfg = OmegaConf.create(cfg.environments[ENV_NAME])

    best_reward, _ = torch.load(
        f"../models/{ENV_NAME}/best-policy.pt", weights_only=False
    )

    # Load replay buffers from file
    rb_exploit = torch.load(f"../models/{ENV_NAME}/rb-exploit.pt", weights_only=False)
    rb_unif = torch.load(f"../models/{ENV_NAME}/rb-unif.pt", weights_only=False)

    cfg_offline = OmegaConf.create(cfg)

    cfg_offline.algorithm.n_steps = MAX_STEPS
    cfg_offline.algorithm.max_epochs = None

    cfg_offline.algorithm.batch_size = BATCH_SIZE
    cfg_offline.algorithm.architecture.actor_hidden_size = NN_ARCHITECTURE
    cfg_offline.algorithm.architecture.critic_hidden_size = NN_ARCHITECTURE
    

    cfg_offline.algorithm.eval_interval = EVAL_INTERVAL
    cfg_offline.algorithm.nb_evals = NB_EVAL_ENVS  # nb of evaluation envs in parallel
    
    cfg_offline.algorithm.save_rb_policy_interval = SAVE_RB_POLICY_INTERVAL

    # learning starts immediately for offline:
    cfg_offline.algorithm.learning_starts = None

    # there is no exploration in offline learning
    cfg_offline.algorithm.action_noise = None
    cfg_offline.algorithm.target_policy_noise = None
    
    if TEST_UNIFORM_PROPORTIONS:
        print("TESTING UNIFORM PROPORTIONS")
        for reward, rb_by_prop in sorted(rb_exploit.items(), reverse=START_WITH_BEST):
            if len(REWARDS_TO_TAKE) > 0 and reward not in REWARDS_TO_TAKE:
                continue
            print(f"Policy with reward {reward} :")
            performances = test_rb_uniform_proportions(
                rb_unif,
                rb_by_prop,
                BUFFER_SIZE,
                PROPORTIONS,
                TD3,
                cfg_offline,
                SEEDS,
            )
            torch.save(performances, f"{OUTPUT_DIR}/uniform_proportions-{ENV_NAME}-scoring-{reward:.0f}")
            
            if FIRST_ONLY:
                print("Skipped intermediate policies")
                break
    
    if TEST_NOISE_LEVELS:
        print("TESTING NOISE LEVELS")
        for reward, rb_by_noise in sorted(rb_exploit.items(), reverse=START_WITH_BEST):
            if len(REWARDS_TO_TAKE) > 0 and reward not in REWARDS_TO_TAKE:
                continue
            print(f"Policy with reward {reward} :")
            performances = test_rb_noise_levels(
                rb_by_noise,
                BUFFER_SIZE,
                TD3,
                cfg_offline,
                SEEDS,
            ) 
            
            torch.save(performances, f"{OUTPUT_DIR}/noise_levels-{ENV_NAME}-scoring-{reward:.0f}")
            
            if FIRST_ONLY:
                print("Skipped intermediate policies")
                break


Testing MountainCarContinuous-v0:
TESTING UNIFORM PROPORTIONS
Policy with reward 61 :


  0%|          | 0/10000 [00:00<?, ?it/s]

KeyboardInterrupt: 